In [2]:
%load_ext autoreload
%autoreload 2
from mg_sampling import PROJECT_DIR
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ast import literal_eval
import itertools
from functools import reduce
from mg_sampling.tools import bayesian_bootstrap, read_plumed

### Motifs

In [3]:
def motif_latex_name(x):
    names = x['inner_names']
    resnames = x['inner_resnames']
    resids = x['inner_resids']
    mgrna = x['mgrna']
    motif = ",".join(f"{resname}^{{{resid}}}_\mathrm{{{name}}}" for resname, resid, name in zip(resnames, resids, names))
    if mgrna:
        motif += '-' + mgrna
    return motif

def motif_name(x):
    names = x['inner_names']
    resnames = x['inner_resnames']
    resids = x['inner_resids']
    mgrna = x['mgrna']
    motif = ",".join(f"{resname}{resid}{name}" for resname, resid, name in zip(resnames, resids, names))
    if mgrna:
        motif += '-' + mgrna
    return motif

def label_disjoint_sets(sets):
    """
    From a list of sets, find group membership such that
    non-disjoint sets belong to the same group.
    """
    from scipy.cluster.hierarchy import DisjointSet
    # Groupping by non-empty intersection
    # In: a list of sets {a, b, c}, {d, e}, {e, f}, {a}, {g, h}...
    # Out: a list of group id such as sets with non-null intersection belong to the same group
    #      0, 1, 1, 0, 2 ...
    # Algo: list elements a, b, c, d, e, f, g, ...
    #       assign different group ids
    #       for each pair of sets, if s1 & s2 != 0, set all elements of s1 | s2 to the leftmost id

    ds = DisjointSet()
    for s in sets:
        for x in s:
            ds.add(x)
            ds.merge(x, s[0])
    
    group_ids = [ds._parents[s[0]] for s in sets]
    _, group_ids = np.unique(group_ids, return_inverse=True)

    return group_ids

In [4]:
def summarize_motifs(df):
    df['motif'] = df.apply(motif_name, axis=1).astype('category')

    # Create a unique id for each block of data for error estimate
    df['block'], _ = pd.factorize(df[['replicate', 'repid']].apply(tuple, axis=1))

    count = df.groupby(['block', 'time', 'motif'], observed=True).size()
    count = count.unstack('motif', fill_value=0)

    p_block = count.groupby('block').mean()

    p = p_block.mean().rename('p')

    # Error estimate: bayesian bootstrap with blocks
    ci = p_block.apply(
        lambda x: tuple(map(float, bayesian_bootstrap(x, np.average).confidence_interval))
    ).transpose().set_axis(['p_low', 'p_high'], axis=1)

    n_block_obs = (p_block > 0).sum().rename('n_block_obs')

    res = pd.concat([p, ci, n_block_obs], axis=1)

    res = res.join(
        df.drop_duplicates(subset='motif', keep='first').set_index('motif')[['n_inner', 'inner_indices', 'inner_names', 'inner_resids', 'inner_resnames', 'stereo', 'mgrna']]
    )

    mg_indices = df.groupby('motif', observed=True)['index'].agg(lambda x: list(set(x))).rename('mg_indices')
    res = res.join(mg_indices)

    return res


In [ ]:
# Replex
t_start = 50000  # ps
t_end = 200000 # ps
systems = ['7ez0', '7ez0_noion']
replicates = [0, 1]
repids = range(8)

for sys in systems:
    df = pd.DataFrame()
    for rep, i in itertools.product(replicates, repids):
        _df = pd.read_parquet(PROJECT_DIR / f'replex/{sys}/mMg_tip3p/replicate{rep}/rep{i+8*rep:02d}/MG_motifs.parquet').assign(replicate=rep, repid=i)
        colvar = read_plumed(PROJECT_DIR / f'replex/{sys}/mMg_tip3p/replicate{rep}/rep{i+8*rep:02d}/bias.COLVAR.{i}')
        colvar['time'] = colvar['time'].astype(int)
        _df['reptemp'] = colvar.set_index('time').loc[_df['time'].astype(int), 'bias.index'].astype(int).values
        _df = _df.query('time >= @t_start and time < @t_end')
        df = pd.concat([df, _df])

    # Multiplexing and analysis
    for λ in range(8):
        res = summarize_motifs(df.query('reptemp == @λ').copy())
        res.to_parquet(PROJECT_DIR / f'replex/{sys}/mMg_tip3p/MG_motifs_summary_lambda{λ}.parquet')

In [ ]:
# Plain MD
t_start = 150000  # ps
t_end = 500000 # ps
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p', 'nMg_tip3p', 'nMg_tip3p_soft']
replicates = range(8)

for sys, ff in itertools.product(systems, forcefields):
    df = pd.DataFrame()
    for rep in replicates:
        try:
            _df = pd.read_parquet(PROJECT_DIR / f'plain_md/{sys}/{ff}/rep{rep:02d}/MG_motifs.parquet').assign(replicate=rep, repid=0)
        except FileNotFoundError as e:
            break
        _df = _df.query('time >= @t_start and time < @t_end')
        df = pd.concat([df, _df])
    else:
        res = summarize_motifs(df)
        res.to_parquet(PROJECT_DIR / f'plain_md/{sys}/{ff}/MG_motifs_summary.parquet')

res

In [ ]:
from ast import literal_eval
# Reference: 7ez0

df = pd.read_csv(PROJECT_DIR / f'structures/7ez0/mMg_tip3p/MG_motifs.csv')
df = df.map(lambda x: literal_eval(x) if isinstance(x, str) and x.startswith(('[','{')) else x)
res = summarize_motifs(df.query('n_inner > 0').copy().assign(replicate=0, repid=0))
res.to_parquet(PROJECT_DIR / f'structures/7ez0/mMg_tip3p/MG_motifs_summary.parquet')


In [ ]:
# Reference: 9cbu

df = pd.read_csv(PROJECT_DIR / f'structures/9cbu/MG_motifs.csv')
df = df.map(lambda x: literal_eval(x) if isinstance(x, str) and x.startswith(('[','{')) else x)
res = summarize_motifs(df.query('n_inner > 0').copy().assign(replicate=0, repid=0))
res.to_parquet(PROJECT_DIR / f'structures/9cbu/MG_motifs_summary.parquet')


In [ ]:
res

### Plot most frequent motifs

In [16]:
# Collect all motifs summaries

res = pd.DataFrame()

# Replex
systems = ['7ez0', '7ez0_noion']
for sys in systems:
    _res = pd.read_parquet(PROJECT_DIR / f'replex/{sys}/mMg_tip3p/MG_motifs_summary_lambda0.parquet').assign(sim=f'replex_{sys}_mMg_tip3p')
    res = pd.concat([res, _res])

# PlainMD
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p', 'nMg_tip3p', 'nMg_tip3p_soft']
for sys, ff in itertools.product(systems, forcefields):
    try:
        _res = pd.read_parquet(PROJECT_DIR / f'plain_md/{sys}/{ff}/MG_motifs_summary.parquet').assign(sim=f'plain_md_{sys}_{ff}')
    except FileNotFoundError:
        pass
    else:
        res = pd.concat([res, _res])

# References
_res = pd.read_parquet(PROJECT_DIR / f'structures/7ez0/mMg_tip3p/MG_motifs_summary.parquet').assign(sim=f'7ez0')
res = pd.concat([res, _res])
# _res = pd.read_parquet(PROJECT_DIR / f'structures/9cbu/MG_motifs_summary.parquet').assign(sim=f'9cbu')
# res = pd.concat([res, _res])
res = res.drop(index="").reset_index()
res['inner_labels'] = res.apply(lambda x: [f"{resname}{resid}{name}" for resname, resid, name in zip(x['inner_resnames'], x['inner_resids'], x['inner_names'])], axis=1)
res['motif_group'] = label_disjoint_sets(res['inner_labels'])

In [28]:
res.set_index(['motif_group', 'motif', 'sim'])

p  \
motif_group motif                                     sim                               
0           A103O1P-Oph                               replex_7ez0_mMg_tip3p  0.273196   
1           A103O2P-Oph                               replex_7ez0_mMg_tip3p  0.144386   
2           A104O1P-Oph                               replex_7ez0_mMg_tip3p  0.062500   
3           A104O2P,U267O2P-cis-2Oph                  replex_7ez0_mMg_tip3p  0.020833   
            A104O2P-Oph                               replex_7ez0_mMg_tip3p  0.259646   
...                                                                               ...   
426         G257O1P,U258O2P,C260O2,U305O2P-Cis-3OphOb 7ez0                   1.000000   
600         U258O1P-Oph                               7ez0                   1.000000   
601         U259O1P-Oph                               7ez0                   1.000000   
108         U305O1P-Oph                               7ez0                   1.000000   
110         U307O2P,A308O2P-cis-2Oph                  7ez0                   1.000000   

                                                                                p_low  \
motif_group motif                                     sim                               
0           A103O1P-Oph                               replex_7ez0_mMg_tip3p  0.156592   
1           A103O2P-Oph                               replex_7ez0_mMg_tip3p  0.043205   
2           A104O1P-Oph                               replex_7ez0_mMg_tip3p  0.003195   
3           A104O2P,U267O2P-cis-2Oph                  replex_7ez0_mMg_tip3p  0.000267   
            A104O2P-Oph                               replex_7ez0_mMg_tip3p  0.138843   
...                                                                               ...   
426         G257O1P,U258O2P,C260O2,U305O2P-Cis-3OphOb 7ez0                   1.000000   
600         U258O1P-Oph                               7ez0                   1.000000   
601         U259O1P-Oph                               7ez0                   1.000000   
108         U305O1P-Oph                               7ez0                   1.000000   
110         U307O2P,A308O2P-cis-2Oph                  7ez0                   1.000000   

                                                                               p_high  \
motif_group motif                                     sim                               
0           A103O1P-Oph                               replex_7ez0_mMg_tip3p  0.452343   
1           A103O2P-Oph                               replex_7ez0_mMg_tip3p  0.308931   
2           A104O1P-Oph                               replex_7ez0_mMg_tip3p  0.184851   
3           A104O2P,U267O2P-cis-2Oph                  replex_7ez0_mMg_tip3p  0.063251   
            A104O2P-Oph                               replex_7ez0_mMg_tip3p  0.416347   
...                                                                               ...   
426         G257O1P,U258O2P,C260O2,U305O2P-Cis-3OphOb 7ez0                   1.000000   
600         U258O1P-Oph                               7ez0                   1.000000   
601         U259O1P-Oph                               7ez0                   1.000000   
108         U305O1P-Oph                               7ez0                   1.000000   
110         U307O2P,A308O2P-cis-2Oph                  7ez0                   1.000000   

                                                                             n_block_obs  \
motif_group motif                                     sim                                  
0           A103O1P-Oph                               replex_7ez0_mMg_tip3p           14   
1           A103O2P-Oph                               replex_7ez0_mMg_tip3p            4   
2           A104O1P-Oph                               replex_7ez0_mMg_tip3p            2   
3           A104O2P,U267O2P-cis-2Oph                  replex_7ez0_mMg_tip3p            1   
            A104O2P-Oph                               repl

In [12]:
res.sort_values('group_max_p', ascending=False)

,p,p_low,p_high,n_block_obs,n_inner,inner_indices,inner_names,inner_resids,inner_resnames,stereo,mgrna,mg_indices,sim,inner_labels,motif_group,group_max_p
motif,,,,,,,,,,,,,,,,
A171O2P-Oph,1.000000,1.000000e+00,1.000000,1,1,[4802],[O2P],[171],[A],None,Oph,[12463],7ez0,[A171O2P],35,1.000000
"A172O2P,A173O2P-cis-2Oph",1.000000,1.000000e+00,1.000000,1,2,"[4835, 4868]","[O2P, O2P]","[172, 173]","[A, A]",cis,cis-2Oph,[12464],7ez0,"[A172O2P, A173O2P]",37,1.000000
U168O2P-Oph,0.247810,1.043615e-01,0.458538,5,1,[4707],[O2P],[168],[U],,Oph,"[269603, 269607, 269616, 269585, 269623]",replex_7ez0_noion_mMg_tip3p,[U168O2P],35,1.000000
U185O2P-Oph,0.050823,2.002883e-02,0.092945,8,1,[5260],[O2P],[185],[U],,Oph,"[269664, 269698, 269700, 269706, 269676, 26955...",replex_7ez0_noion_mMg_tip3p,[U185O2P],41,1.000000
"G272O1P,A301O1P-cis-2Oph",0.001446,4.009415e-05,0.003974,1,2,"[8043, 8969]","[O1P, O1P]","[272, 301]","[G, A]",cis,cis-2Oph,[269599],replex_7ez0_noion_mMg_tip3p,"[G272O1P, A301O1P]",80,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A38H5'2-X,0.000036,1.345599e-06,0.000141,1,1,[527],[H5'2],[38],[A],,X,[12448],plain_md_7ez0_nMg_tip3p_soft,[A38H5'2],152,0.000036
A70H61-X,0.000036,9.318712e-07,0.000112,1,1,[1563],[H61],[70],[A],,X,[269887],plain_md_7ez0_mMg_tip3p,[A70H61],182,0.000036
C121H5'2-X,0.000036,1.567014e-06,0.000122,1,1,[3202],[H5'2],[121],[C],,X,[269783],plain_md_7ez0_mMg_tip3p,[C121H5'2],208,0.000036


In [ ]:
sorter = list(motifs)
out = res.loc[res['motif'].isin(motifs)].copy()
out['motif'] = out['motif'].astype('category')
out = out.groupby(['motif', 'sim'], observed=False)['proba'].sum().unstack().loc[sorter].stack()
out.loc[out == 0] = np.nan


In [ ]:
out

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.scatter(
    np.arange(len(motifs)).repeat(len(sims)),
    np.tile(np.arange(len(sims)), len(motifs)),
    c=out,
    marker='s',
    edgecolors='black',
    s=100,
    cmap='magma_r',
)
ax.invert_yaxis()
ax.set_aspect(1)
ax.set_yticks(np.arange(len(sims)), labels=sims)
ax.set_xticks(np.arange(len(motifs)), labels=motifs, rotation=45, ha='left')
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

In [ ]:
out.max()

In [ ]:
sns.barplot(
    x,
    x='n_inner',
    y='%',
    errorbar=_agg,
    capsize=0.1,
)

In [ ]:
def _plot(data, **kws):
    x = data.groupby(['frame', 'n_inner]').size()
    res = bayesian_bootstrap(x, np.average)

In [ ]:
# Replex
systems = ['7ez0']
forcefields = ['mMg_tip3p']
replicates = [0]
repids = [0, 7]
ions = ['MG']

df = pd.DataFrame()
for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'replicate{replicate}' / f'rep{repid:02d}'
    _df = pd.read_csv(path / f'{ion}_OW_motifs.csv', na_filter=False, usecols=['time', 'index', 'n_inner', 'mgrna'])
    _df['system'] = system
    _df['forcefield'] = forcefield
    _df['repid'] = int(repid)
    _df['replicate'] = replicate
    _df['method'] = 'replex'
    df = pd.concat([df, _df])

# PlainMD
systems = ['7ez0']
forcefields = ['mMg_tip3p', 'nMg_tip3p']
replicates = range(8)
ions = ['MG']

for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    try:
        path = PROJECT_DIR / 'plain_md' / system / forcefield / f'rep{replicate:02d}'
        _df = pd.read_csv(path / f'{ion}_OW_motifs.csv', na_filter=False, usecols=['time', 'index', 'n_inner', 'mgrna'])
        _df['system'] = system
        _df['forcefield'] = forcefield
        _df['replicate'] = replicate
        _df['method'] = 'plain_md'
        df = pd.concat([df, _df])
    except FileNotFoundError as e:
        print(e)
        pass

for col in ['mgrna', 'forcefield', 'system', 'method']:
    df[col] = df[col].astype('category')

In [ ]:
df['repid'] = df['repid'].fillna(0).astype(int)
df['sim'] = df['method'].str.cat(df['forcefield'], sep='_').str.cat(df['repid'].astype(str), sep='_')
df['sim'] = df['sim'].map({
    'replex_mMg_tip3p_0': 'μMg (REx, $λ_0$)',
    'replex_mMg_tip3p_7': 'μMg (REx, $λ_7$)',
    'plain_md_mMg_tip3p_0': 'μMg (MD)',
    'plain_md_nMg_tip3p_0': 'nMg (MD)',
})

In [ ]:
df.head()

In [ ]:
palette = ['tomato', 'darkred', 'gray', 'mediumpurple']
labels = ['μMg (REx, $λ_0$)', 'μMg (REx, $λ_7$)', 'μMg (MD)', 'nMg (MD)']

tstart_replex = 50000  # ps
tstart_plain_md = 150000
grouped = df.query(f"(method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})")

sns.set_style("ticks", {'axes.grid': True})



In [ ]:
palette = ['tomato', 'darkred', 'gray', 'mediumpurple']
labels = ['μMg (REx, $λ_0$)', 'μMg (REx, $λ_7$)', 'μMg (MD)', 'nMg (MD)']

tstart_replex = 50000  # ps
tstart_plain_md = 150000
grouped = df.query(f"(method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})")

sns.set_style("ticks", {'axes.grid': True})

ax = sns.histplot(grouped,
    y='n_inner',
    hue='sim',
    hue_order=labels,
    palette=palette,
    stat='percent',
    discrete=True,
    multiple='dodge',
    common_norm=False,
)

# ax.add_legend(g._legend_data, loc='center', bbox_to_anchor=(0.5, 1), ncol=len(labels))
ax.set_xscale('log')
ax.set_xlabel("[%]")
ax.set_ylabel(r"Coordination")
plt.gcf().set_size_inches((5, 5))
plt.tight_layout()
plt.savefig('figures/average_n_inner.pdf', bbox_inches='tight')

### Inner-bound (< 2.8 Å)

In [ ]:
# Replex
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p']
replicates = [0]
repids = [0]
ions = ['MG', 'NA']

df = []
for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'replicate{replicate}' / f'rep{repid:02d}'
    _df = pd.read_csv(path / f'{ion}_motifs.csv', na_filter=False)
    _df = _df.loc[_df['time'] % 1000.0 == 0]
    _df['system'] = system
    _df['forcefield'] = forcefield
    _df['repid'] = repid
    _df['replicate'] = replicate
    _df['method'] = 'replex'
    df.append(_df)

# PlainMD
systems = ['7ez0', '7ez0_noion']
forcefields = ['nMg_tip3p_hard', 'mMg_tip3p_hard', 'nMg_tip3p']
replicates = range(8)
ions = ['MG', 'NA']

for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    try:
        path = PROJECT_DIR / 'plain_md' / system / forcefield / f'rep{replicate:02d}'
        _df = pd.read_csv(path / f'{ion}_motifs.csv', na_filter=False)
        _df = _df.loc[_df['time'] % 1000.0 == 0]
        _df['system'] = system
        _df['forcefield'] = forcefield
        _df['replicate'] = replicate
        _df['method'] = 'plain_md'
        df.append(_df)
    except FileNotFoundError as e:
        print(e)
        pass

df = pd.concat(df)
for col in ['name', 'mgrna', 'forcefield', 'system', 'method']:
    df[col] = df[col].astype('category')


In [ ]:
tstart_replex = 50000  # ps
tstart_plain_md = 150000

grouped = df.query(f"(method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})").groupby(
    ['time', 'name', 'n_inner', 'forcefield', 'system', 'replicate', 'method'], observed=True
    ).size().reset_index(name='counts')

In [ ]:
# Compute unbound by taking the complement
n_tot = {'MG': 178, 'NA': 121}
tot = grouped.groupby(['time', 'name', 'forcefield', 'system', 'replicate', 'method'], observed=True).sum()['counts'].reset_index()
tot['tot'] = tot['name'].map(n_tot).astype(int)
tot['counts'] = tot['tot'] - tot['counts']
tot = tot.drop(columns='tot')
tot.insert(2, 'n_inner', -1)
grouped = pd.concat([grouped, tot])
grouped['sim'] = grouped['method'].str.cat(grouped['forcefield'], sep='_')
grouped['sim'] = grouped['sim'].map({
    'replex_mMg_tip3p': 'μMg (REx)',
    'plain_md_mMg_tip3p_hard': 'μMg (MD)',
    'plain_md_nMg_tip3p_hard': 'nMg (MD)',
    'plain_md_nMg_tip3p': 'nMg (MD-soft)',
})
grouped['binding'] = grouped['n_inner'].replace(-1, 'Bulk').replace(0, 'Outer')
grouped = grouped.sort_values('n_inner')

In [ ]:
grouped.query("forcefield == 'nMg_tip3p'")

In [ ]:
palette = ['tomato', 'gray', 'mediumpurple', 'mediumpurple']
labels = ['μMg (REx)', 'μMg (MD)', 'nMg (MD)', 'nMg (MD-soft)']

g = sns.FacetGrid(grouped, row='name', col='system', sharey='row')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    y='binding',
    x='counts',
    orient='h',
    hue='sim',
    hue_order=labels,
    palette=palette,
)

g.add_legend(g._legend_data, loc='center', bbox_to_anchor=(0.5, 1), ncol=len(labels))
g.set_xlabels("Avg. number")
g.set_ylabels(r"Coordination")
g.set_titles("{row_name} ({col_name})")
plt.gcf().set_size_inches((5, 5))
plt.tight_layout()
plt.savefig('figures/average_coordination.pdf', bbox_inches='tight')

## Inner-bound sites

In [ ]:
tstart_replex = 50000  # ps
tstart_plain_md = 150000

grouped = df.query(f"((method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})) and n_inner > 0").copy()
grouped['inner_names'] = grouped['inner_names'].map(literal_eval)

In [ ]:
grouped = grouped.explode('inner_names')

In [ ]:
# Count by name
grouped = grouped.groupby(['name', 'system', 'method', 'forcefield', 'inner_names'], observed=True).size()


In [ ]:
grouped.groupby(['name', 'system', 'method', 'forcefield'], observed=True).sum()

In [ ]:
grouped = (grouped / grouped.groupby(['name', 'system', 'method', 'forcefield'], observed=True).sum())

In [ ]:
grouped = grouped.reset_index(name='proportion')

In [ ]:
grouped

In [ ]:
grouped['sim'] = grouped['method'].str.cat(grouped['forcefield'], sep='_')
grouped['sim'] = grouped['sim'].map({
    'replex_mMg_tip3p': 'μMg (REx)',
    'plain_md_mMg_tip3p_hard': 'μMg (MD)',
    'plain_md_nMg_tip3p_hard': 'nMg (MD)',
})
palette = ['tomato', 'gray', 'mediumpurple']
labels = ['μMg (REx)', 'μMg (MD)', 'nMg (MD)']

g = sns.FacetGrid(grouped, row='name', col='system', sharey='row')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    y='inner_names',
    x='proportion',
    orient='h',
    hue='sim',
    hue_order=labels,
    palette=palette,
    order=grouped[grouped['proportion'] > 0.01].sort_values('proportion', ascending=False)['inner_names'].unique(),
)

g.add_legend(g._legend_data, loc='center', bbox_to_anchor=(0.5, 1), ncol=len(labels))
g.set_xlabels("Proportion")
g.set_ylabels("Inner-bound atom type")
g.set_titles("{row_name} ({col_name})")
for axis in g.axes.flat:
    axis.tick_params(labelleft=True)
plt.gcf().set_size_inches((5, 5))
plt.tight_layout()
plt.savefig('figures/binding_sites.pdf', bbox_inches='tight')

### Summarize motifs